# 🧠 SkillBridge AI: ETL Pipeline Tutorial
Notebook ini adalah panduan pembelajaran untuk memahami proses **Ekstraksi, Transformasi, dan Loading (ETL)** data lowongan kerja menggunakan **Gemma 2B via Ollama**.

### Tech Stack:
- **Python** (Logic)
- **Ollama** (Local LLM Server)
- **Gemma 2B** (Model)
- **Pydantic** (Data Validation)

## 1. Setup & Inisialisasi
Kita akan memuat environment variables dan mendefinisikan skema data yang kita harapkan.

In [ ]:
import os
import json
import ollama
import re
import pandas as pd
from pydantic import BaseModel, Field
from typing import List
from dotenv import load_dotenv

load_dotenv("../.env")

RAW_DATA_PATH = "../data/raw/google_jobs_results.json"
OLLAMA_MODEL = "gemma2:2b"

class JobExtraction(BaseModel):
    is_valid_job: bool
    required_skills: List[str]

print(f"Using model: {OLLAMA_MODEL}")

Using model: gemma2:2b


## 2. Load Raw Data
Mari kita lihat data mentah yang kita miliki.

In [ ]:
with open(RAW_DATA_PATH, 'r', encoding='utf-8') as f:
    raw_jobs = json.load(f)

print(f"Total data mentah: {len(raw_jobs)} lowongan.")
print("Contoh data pertama:")
print(json.dumps(raw_jobs[0], indent=2)[:500] + "...")

Total data mentah: 4911 lowongan.
Contoh data pertama:
{
  "title": "Lowongan Kerja Visual Merchandiser (36hpw) - Jakarta Timur",
  "company_name": "Forever 21",
  "location": "Jakarta Timur, Kota Jakarta Timur, Daerah Khusus Ibukota Jakarta",
  "via": "Part-Time Sabtu Minggu",
  "share_link": "https://www.google.co.id/search?ibp=htl;jobs&q=lowongan+kerja&htidocid=L-pGNAHNTwXDqHvKAAAAAA%3D%3D&hl=id-ID&shem=rimspwouoe&shndl=37&shmd=H4sIAAAAAAAA_12OwWoCQQyG6dVH6ClHW3TXWvSgV1F0a_EgvUp2DTvjjpMlmak-to_QURC0l_An5P_4OpeXDn7xiX2NHgqSA8KP1YgO1iSVQb-3SgLdz7Fp...


In [ ]:
df = pd.DataFrame(raw_jobs)
df_clean = df.drop_duplicates(subset=['title', 'company_name'], keep='first')
print(f"Jumlah data sebelum deduplikasi: {len(df)}")
print(f"Jumlah data setelah deduplikasi: {len(df_clean)}")
df_raw = df_clean.to_dict('records')

Jumlah data sebelum deduplikasi: 4911
Jumlah data setelah deduplikasi: 1868


## 3. Layer 1: Local Heuristic (Denoising)
Sebelum mengirim ke AI, kita filter dulu lowongan yang sudah pasti spam atau tidak valid agar lebih cepat.

In [ ]:
import re  

PHONE_RE = re.compile(r'\b0\d{8,12}\b')

def is_potential_job(title: str, desc: str) -> bool:
    """
    Heuristic filter SEBELUM memanggil AI.
    Tujuan: Buang data sampah secara murah (tanpa API call).
    Hemat 20-40% API call yang tidak perlu.
    """
    if len(title.strip()) < 5:
        return False
    if len(desc.strip()) < 50:
        return False
    
    text = (title + " " + desc).lower()
    
    spam_keywords = [
        "resep masakan", "cara memasak", "film sub", "download mp3",
        "baca selengkapnya", "manfaat buah", "prediksi togel", "judi online"
    ]
    
    if any(k in text for k in spam_keywords):
        return False

    
    return True


# 4. Layer 2: AI Extraction (Ollama + Gemma 2B)
Di sini kita menggunakan **Few-Shot Prompting** untuk melakukan ekstraksi fitur yang ada pada deskripsi pekerjaan. Kita berikan contoh agar model paham format JSON yang kita inginkan.

In [ ]:
def build_prompt(title: str, company: str, desc: str) -> str:
    # ✅ Potong 750 karakter  — Gemma 2B tidak membaca lebih jauh secara efektif
    SHORT_DESC = desc[:750].strip()

    return f"""Parse job listing to JSON. Output ONLY valid JSON, no other text.

Schema:
{{"is_valid_job":bool,"hard_skills":[str],"soft_skills":[str,max3],"education_level":"SD"|"SMP"|"SMA"|"SMK"|"D3"|"S1"|"S2"|"S3"|null,"min_experience_years":int,"certifications":[str],"job_category":"Technology"|"Creative & Design"|"Finance & Accounting"|"Marketing & Sales"|"Food & Beverage"|"Operations & Logistics"|"Administration"|"Healthcare"|"Education"|"Engineering"|"Legal"|"Customer Service"|"Human Resources"|"Lainnya","job_subcategory":str,"seniority_level":"Entry"|"Junior"|"Mid"|"Senior"|"Lead"|"Manager","work_arrangement":"Onsite"|"Remote"|"Hybrid","employment_type":"Full-time"|"Part-time"|"Contract"|"Internship"|"Freelance","salary_min":int,"salary_max":int,"is_multi_position":bool,"job_responsibilities":[str,max3],"cleaned_title":str}}

Rules: hard_skills=technical only. salary=0 unless EXPLICITLY stated. cleaned_title=no "Lowongan Kerja"/city noise. is_valid_job=false ONLY IF it is an article/tutorial, but TRUE if it's a real job post even if it has minimal info. Extract employment_type accurately from text (e.g., Freelance, Part-time).For education_level, extract from phrases like "Educational Level: SMA", "Pendidikan minimal D3", "Lulusan S1". If it says "SMA - SMK" or "SMA/SMK", use "SMA".

Example 1:
{{"is_valid_job":true,"hard_skills":["Microsoft Office","Pengelolaan Limbah B3"],"soft_skills":["Komunikasi","Teliti"],"education_level":"SMK","min_experience_years":1,"certifications":["OPLB3"],"job_category":"Engineering","job_subcategory":"Waste Management","seniority_level":"Junior","work_arrangement":"Onsite","employment_type":"Full-time","salary_min":4000000,"salary_max":5000000,"is_multi_position":false,"job_responsibilities":["Pengelolaan limbah B3","Laporan berkala","Mematuhi K3"],"cleaned_title":"Waste Management Operator"}}

Example 2 (Minimalist / Freelance):
Job: Loker Freelance Desain Grafis WA 08123456
Description: Dibutuhkan segera freelance desain grafis, kerja dari rumah. Hubungi WA 08123456
JSON:
{{"is_valid_job":true,"hard_skills":["Desain Grafis"],"soft_skills":[],"education_level":null,"min_experience_years":0,"certifications":[],"job_category":"Creative & Design","job_subcategory":"Graphic Designer","seniority_level":"Entry","work_arrangement":"Remote","employment_type":"Freelance","salary_min":0,"salary_max":0,"is_multi_position":false,"job_responsibilities":[],"cleaned_title":"Desain Grafis"}}

Example 3 (Minimalist & Education Extraction):
Job: Loker Freelance Operator Produksi PT Yamaha Motor | PT Yamaha
Description: Educational Level : SMA - SMK. Area Placement : Jakarta. Requirement: Pengalaman 0-1 Tahun. Fresh Graduate dipersilakan melamar. Cekatan, Teliti. Hubungi WA 08123456
JSON:
{{"is_valid_job":true,"hard_skills":["Operator Produksi"],"soft_skills":["Cekatan","Teliti"],"education_level":"SMA","min_experience_years":0,"certifications":[],"job_category":"Operations & Logistics","job_subcategory":"Production","seniority_level":"Entry","work_arrangement":"Onsite","employment_type":"Freelance","salary_min":0,"salary_max":0,"is_multi_position":false,"job_responsibilities":["Operator Produksi"],"cleaned_title":"Operator Produksi"}}

Job: {title} | {company}
Description: {SHORT_DESC}
JSON:""".strip()



In [ ]:
# Definisikan di level MODUL, bukan di dalam fungsi
_SKILL_BLACKLIST   = {"sma", "smk", "d3", "s1", "s2", "s3", "sd", "smp", "prodi", "helo", "halo", "fresh graduate"}
_VALID_EDU         = {"SD", "SMP", "SMA", "SMK", "D3", "S1", "S2", "S3", None}
_VALID_CATEGORY    = {"Technology", "Creative & Design", "Finance & Accounting", "Marketing & Sales", "Food & Beverage", "Operations & Logistics", "Administration", "Healthcare", "Education", "Engineering", "Legal", "Customer Service", "Human Resources", "Lainnya"}
_VALID_SENIORITY   = {"Entry", "Junior", "Mid", "Senior", "Lead", "Manager"}
_VALID_ARRANGEMENT = {"Onsite", "Remote", "Hybrid"}
_VALID_EMPLOYMENT  = {"Full-time", "Part-time", "Contract", "Internship", "Freelance"}


async def process_with_ai(job: dict) -> dict | None:
    prompt = build_prompt(
        job['title'],
        job.get('company_name', 'Unknown'),
        job.get('description', '')
    )

    # ✅ PERBAIKAN: Seluruh blok (termasuk ollama.chat) masuk try/except
    try:
        response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            format='json',
            options={
                'temperature': 0,       # ✅ Turun dari 0.1 → deterministic, lebih cepat
                'num_predict': 400,     # ✅ Turun dari 768 → cukup untuk output JSON ini
                'num_ctx': 1536,        # ✅ Batasi context window eksplisit
            }
        )
        data = json.loads(response['message']['content'])

    except json.JSONDecodeError as e:
        print(f"[JSON Error] {job.get('title', '?')}: {e}")
        return None
    except Exception as e:
        # Menangkap error Ollama (timeout, model crash, dll.)
        print(f"[Ollama Error] {job.get('title', '?')}: {e}")
        return None

    # === Defaults ===
    data.setdefault('hard_skills', [])
    data.setdefault('soft_skills', [])
    data.setdefault('education_level', None)
    data.setdefault('min_experience_years', 0)
    data.setdefault('certifications', [])
    data.setdefault('job_category', 'Lainnya')
    data.setdefault('job_subcategory', 'Lainnya')
    data.setdefault('seniority_level', 'Entry')
    data.setdefault('work_arrangement', 'Onsite')
    data.setdefault('employment_type', 'Full-time')
    data.setdefault('salary_min', 0)
    data.setdefault('salary_max', 0)
    data.setdefault('is_multi_position', False)
    data.setdefault('job_responsibilities', [])
    data.setdefault('cleaned_title', job.get('title', ''))

    # === Filter skills (pakai konstanta modul, bukan buat set baru) ===
    data["hard_skills"] = [s for s in data["hard_skills"] if s.lower() not in _SKILL_BLACKLIST and len(s) > 2]
    data["soft_skills"]  = [s for s in data["soft_skills"]  if s.lower() not in _SKILL_BLACKLIST and len(s) > 2]

    # === Validasi enum ===
    if data.get("education_level") not in _VALID_EDU:        data["education_level"] = None
    if data.get("job_category") not in _VALID_CATEGORY:      data["job_category"] = "Lainnya"
    if data.get("seniority_level") not in _VALID_SENIORITY:  data["seniority_level"] = "Entry"
    if data.get("work_arrangement") not in _VALID_ARRANGEMENT: data["work_arrangement"] = "Onsite"
    if data.get("employment_type") not in _VALID_EMPLOYMENT:  data["employment_type"] = "Full-time"

    # === Validasi numerik ===
    data["salary_min"]           = int(data.get("salary_min") or 0)
    data["salary_max"]           = int(data.get("salary_max") or 0)
    data["min_experience_years"] = int(data.get("min_experience_years") or 0)
    data["is_multi_position"]    = bool(data.get("is_multi_position", False))

    return data


# 5. Simulasi Running ETL
Kita coba jalankan untuk 3 data pertama.

In [ ]:
import asyncio

async def demo():
    for i in range(3):
        job = df_raw[i]
        if is_potential_job(job['title'], job.get('description', '')):
            result = await process_with_ai(job)
            print(f"--- Lowongan {i+1} ---")
            print(f"Judul: {job['title']}")
            print(f"AI Result: {result}")
            print("\n")

await demo()

--- Lowongan 1 ---
Judul: Lowongan Kerja Visual Merchandiser (36hpw) - Jakarta Timur
AI Result: {'is_valid_job': True, 'hard_skills': ['Visual Merchandising'], 'soft_skills': ['Kreativitas', 'Teliti'], 'education_level': None, 'min_experience_years': 0, 'certifications': [], 'job_category': 'Creative & Design', 'job_subcategory': 'Visual Merchandiser', 'seniority_level': 'Entry', 'work_arrangement': 'Onsite', 'employment_type': 'Full-time', 'salary_min': 0, 'salary_max': 0, 'is_multi_position': False, 'job_responsibilities': ['Penataan display produk', 'Menjaga standar visual merchandising brand', 'Mengatur layout toko', 'Membantu promosi seasonal'], 'cleaned_title': 'Visual Merchandiser'}


--- Lowongan 2 ---
Judul: LOKER BULAN MEI 2026 UNTUK DI BEKASI SEGERA | LOWONGAN KERJA TERBARU UNTUK DI BEKASI
AI Result: {'is_valid_job': True, 'hard_skills': ['Operator Produksi'], 'soft_skills': [], 'education_level': 'SMA', 'min_experience_years': 0, 'certifications': [], 'job_category': 'Food 

In [ ]:
output_dir = "../data/cleaned"
output_file = os.path.join(output_dir, "cleaned_jobs.json")
os.makedirs(output_dir, exist_ok=True)

async def save_cleaned_data():
    cleaned_list = []

    # 1. Cek apakah file sudah ada. Jika ada, LOAD datanya untuk dilanjutkan (Resume)
    if os.path.exists(output_file):
        with open(output_file, "r", encoding="utf-8") as f:
            try:
                cleaned_list = json.load(f)
                print(f"Melanjutkan dari {len(cleaned_list)} data yang sudah ada...")
            except json.JSONDecodeError:
                cleaned_list = [] # Jika file kosong/rusak
    else:
        # Buat file baru jika belum ada
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(cleaned_list, f, indent=2, ensure_ascii=False)

    # 2. Buat Set berisi 'job_id' yang sudah diproses agar kita bisa skip (mencegah duplikat)
    # Jika tidak ada job_id, kita pakai share_link atau title sebagai patokan
    processed_ids = {job.get('job_id', job.get('title')) for job in cleaned_list}

    for i in range(len(df_raw)):
        job = df_raw[i]
        job_identifier = job.get('job_id', job.get('title'))

        # 3. CEK DUPLIKAT: Jika data sudah ada di JSON, LEWATI (Skip)
        if job_identifier in processed_ids:
            continue

        # Jika belum ada, proses dengan AI
        if is_potential_job(job['title'], job.get('description', '')):
            result = await process_with_ai(job)

            if result:
                job.update(result)
                cleaned_list.append(job)

                # Masukkan ke daftar processed agar tidak dobel
                processed_ids.add(job_identifier)

                with open(output_file, "w", encoding="utf-8") as f:
                    json.dump(cleaned_list, f, indent=2, ensure_ascii=False)

                print(f"[{len(cleaned_list)}] Berhasil memproses dan menyimpan: {job['title']}")

    print(f"Selesai! Semua data berhasil diproses dan disimpan di: {output_file}")

# Jalankan fungsinya
await save_cleaned_data()


Melanjutkan dari 1447 data yang sudah ada...
[1448] Berhasil memproses dan menyimpan: Administrator Dropship Freelance – TANPA IJAZAH IRT, Balikpapan
[1449] Berhasil memproses dan menyimpan: Mitra Kurir Motor - Balikpapan
[1450] Berhasil memproses dan menyimpan: Resmi Lowongan Kerja Administrasi Proyek
[1451] Berhasil memproses dan menyimpan: Loker LOADER PABRIK BALIKPAPAN, Lulusan SMA, Langsung diterima
[1452] Berhasil memproses dan menyimpan: Supervisor Finance Accounting Tax at PT Andindo Duta Perkasa
[1453] Berhasil memproses dan menyimpan: Loker Dari Rumah 3-5 JAM per Hari, Balikpapan
[1454] Berhasil memproses dan menyimpan: Lowongan Kerja Staf / Administrasi umum, Pekerjaan
[1455] Berhasil memproses dan menyimpan: Lowongan Kerja PT. KIMIA FARMA Tbk, Samarinda, Kalimantan TimurPT Kimia Farma (Persero) Tbk Lowongan Kerja PT. KIMIA FARMA Tbk
[1456] Berhasil memproses dan menyimpan: Lowongan Kerja Guru Bahasa Mandarin di Balikpapan
[1457] Berhasil memproses dan menyimpan: Brand & Eve

# 6. Data Refinement 

In [ ]:
import json
import re

def refine_job_data(raw_jobs):
    refined_data = []
    
    for job in raw_jobs:
        # 1. Skip data yang tidak valid
        if not job.get("is_valid_job", True):
            continue
            
        extensions_str = " ".join(job.get("extensions", [])).lower()
        
        # 2. Ekstrak & Konversi Gaji
        # Cek jika AI gagal mengekstrak gaji (bernilai 0), maka kita fallback ke regex
        if job.get("salary_min") == 0 and job.get("salary_max") == 0:
            # Mencari berbagai format gaji: Rp 5,2 jt, Rp 150 rb, Rp 146.350, Rp 3.000.000
            matches = re.findall(r"rp\s*(\d{1,3}(?:\.\d{3})+|\d+[\.,]\d+|\d+)\s*(jt|juta|rb|ribu)?", extensions_str)
            if matches:
                vals = []
                for val_str, suffix in matches:
                    try:
                        if suffix in ["jt", "juta"]:
                            # Format desimal: 5,2 jt atau 5.2 jt -> hilangkan koma jadi titik
                            num = float(val_str.replace(",", ".")) * 1_000_000
                        else:
                            # Format ribuan asli: 146.350 -> hapus titik, jadikan float
                            num = float(val_str.replace(".", "").replace(",", "."))
                            if suffix in ["rb", "ribu"]:
                                num *= 1_000
                        vals.append(num)
                    except ValueError:
                        continue
                
                if vals:
                    # Asumsi 22 hari kerja per bulan jika "per hari"
                    # Asumsi 4 minggu kerja per bulan jika "per minggu"
                    multiplier = 1
                    if "per hari" in extensions_str or "perhari" in extensions_str:
                        multiplier = 22
                    elif "per minggu" in extensions_str or "perminggu" in extensions_str:
                        multiplier = 4
                    elif "per jam" in extensions_str or "perjam" in extensions_str:
                        multiplier = 4
                    else:
                        multiplier = 1
                    job["salary_min"] = int(min(vals) * multiplier)
                    job["salary_max"] = int(max(vals) * multiplier)

        # 3. Koreksi Anomali Pengalaman (Misal 722 hari jadi 2 tahun)
        exp = job.get("min_experience_years", 0)
        if exp > 50:
            job["min_experience_years"] = int(exp / 365)
            
        # 4. Standarisasi Tipe Pekerjaan
        desc_lower = job.get("description", "").lower()
        
        if "freelance" in extensions_str or "lepas" in extensions_str or "freelance" in desc_lower:
            job["employment_type"] = "Freelance"
        elif "magang" in extensions_str or "internship" in extensions_str or "intern" in desc_lower:
            job["employment_type"] = "Internship"
        elif "paruh waktu" in extensions_str or "part time" in extensions_str or "paruh waktu" in desc_lower:
            job["employment_type"] = "Part-time"
        elif "tetap" in extensions_str or "full time" in extensions_str or "karyawan tetap" in desc_lower:
            job["employment_type"] = "Full-time"
        elif "kontrak" in extensions_str or "contract" in extensions_str or "kontrak" in desc_lower:
            job["employment_type"] = "Contract"

        # 5. Normalisasi Skills (Lowercase, hapus yang lebih dari 4 kata)
        job["hard_skills"] = list(set([
            s.strip().lower() for s in job.get("hard_skills", []) 
            if len(s.split()) <= 4 and len(s) > 2
        ]))
        
        job["soft_skills"] = list(set([
            s.strip().lower() for s in job.get("soft_skills", []) 
            if len(s.split()) <= 3 and len(s) > 2
        ]))
        
        job["original_salary_str"] = next((s for s in job.get("extensions", []) if "rp" in s.lower()), None)
        refined_data.append(job)
        
    return refined_data


In [ ]:
# 1. Baca data hasil AI
with open("../data/cleaned/cleaned_jobs.json", "r", encoding="utf-8") as f:
    jobs_from_ai = json.load(f)

# 2. Bersihkan datanya
final_ready_jobs = refine_job_data(jobs_from_ai)

# 3. Simpan kembali untuk digunakan oleh SBERT/FAISS
with open("../data/cleaned/refined_jobs.json", "w", encoding="utf-8") as f:
    json.dump(final_ready_jobs, f, indent=2, ensure_ascii=False)

print(f"Data berhasil dibersihkan! Tersisa {len(final_ready_jobs)} loker valid.")